# Day 07: CUDA Kernels, Kernel Selection & Kernel Fusion(Day 07:CUDA kernel、kernel 选择与 kernel 融合)
> *100 Days of Inference* | Layer: **Runtime** | Book: *Inference Engineering* Ch 4.1 (pp. 96–101)
> (《100 天学推理》系列 | 层级:**运行时** | 对应《Inference Engineering》第 4.1 章,第 96–101 页)

**Prerequisite:** Day 06 (ops:byte ratio)

> **前置要求:**Day 06(算力字节比)

**Goal:** Understand what a CUDA kernel is, how kernel selection works, and why kernel fusion is critical for memory-bound inference.

> **目标:**理解什么是 CUDA kernel、kernel 选择如何运作,以及为什么 kernel 融合对带宽受限的推理至关重要。

## What problem does this solve?(这一讲解决什么问题?)

Day 04 showed that decode is memory-bound: the bottleneck is reading bytes from GPU memory, not computing. So why does compute efficiency still matter?

> Day 04 讲过,decode 是带宽受限的:瓶颈在从显存读字节,而不在计算。那为什么计算效率依然重要?

Because the *way* you move bytes matters as much as *how many* you move. When two operations run back-to-back — say, a matrix multiply followed by an activation function — the naive approach writes the intermediate result to GPU memory and reads it back for the next operation. That's wasted memory traffic.

> 因为*搬运字节的方式*与*搬运多少字节*同样重要。当两个操作首尾相接——比如一次矩阵乘后面跟一个激活函数——朴素做法会把中间结果写回显存,再为下一个操作读回来。这就是浪费的内存流量。

CUDA kernels are the functions that run on the GPU. Understanding them is like understanding how system calls work: you don't write them every day, but knowing what's happening under the hood explains why things are fast or slow. Kernel selection and fusion are the two biggest levers for reducing unnecessary memory traffic.

> CUDA kernel 就是在 GPU 上运行的函数。理解它们就像理解系统调用:你不是天天写它,但知道底层发生了什么,才能解释事情为什么快、为什么慢。kernel 选择与融合是减少不必要内存流量的两个最大杠杆。

## Concept Overview(概念总览)

**CUDA** (Compute Unified Device Architecture) is NVIDIA's platform for running code on GPUs. A **CUDA kernel** is a function that runs in parallel across thousands of GPU threads simultaneously.

> **CUDA**(计算统一设备架构)是 NVIDIA 在 GPU 上运行代码的平台。**CUDA kernel** 是一个由数千个 GPU 线程同时并行运行的函数。

**Infrastructure analogy:** Think of a CUDA kernel like a shell script that runs on every core simultaneously. Your script processes one item; the GPU runs it on 4096 items at once.

> **基础设施类比:**可以把 CUDA kernel 想成一个同时在每个核心上运行的 shell 脚本。你的脚本只处理一个元素,GPU 却让 4096 个元素同时跑它。

Key CUDA components:
- **Kernel:** The parallel function itself (like a thread handler)
- **CUDA driver:** The low-level interface to the GPU hardware (like a device driver)
- **CUDA runtime:** Developer-facing API for launching kernels (like a system call API)
- **cuBLAS:** Pre-built kernels for linear algebra (GEMM = General Matrix-Matrix Multiply)
- **CUTLASS:** Template library for writing high-performance kernels

> CUDA 关键组件:
> - **Kernel:**并行函数本身(类似线程处理函数)
> - **CUDA driver:**对接 GPU 硬件的底层接口(类似设备驱动)
> - **CUDA runtime:**面向开发者的 kernel 启动 API(类似系统调用接口)
> - **cuBLAS:**预编译的线性代数 kernel(GEMM = 通用矩阵乘)
> - **CUTLASS:**编写高性能 kernel 的模板库

**Kernel fusion:** Combining two kernels into one to eliminate the intermediate memory read/write — reduces total bytes moved.

> **kernel 融合(kernel fusion):**把两个 kernel 合并为一个,消除中间结果的内存读写——减少总搬运字节数。

In [1]:
!pip install -q numpy matplotlib torch 2>/dev/null
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10
GPU Memory: 128.5 GB


## Part 1: What a CUDA Kernel Is(第一部分:CUDA kernel 是什么)

A kernel is just a function. Here is the simplest possible one from the book:

> kernel 不过是一个函数。下面是书里给出的最简单的一个:

```c
__global__ void doubleArray(float *arr, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        arr[i] *= 2.0f;
    }
}
```

> (C 代码大意:把数组每个元素翻倍;`i = blockIdx.x * blockDim.x + threadIdx.x` 是全局线程索引公式,决定每个线程负责哪个元素)

This multiplies every element of an array by 2. On a CPU, this runs element-by-element (O(n) time). On a GPU, all elements run simultaneously (O(1) effective time for large n, limited by memory bandwidth).

> 它把数组每个元素乘 2。在 CPU 上要逐个元素跑(O(n) 时间);在 GPU 上所有元素同时跑(对大 n 相当于 O(1),受限于内存带宽)。

The `__global__` keyword means it runs on the GPU. The line `int i = blockIdx.x * blockDim.x + threadIdx.x` is the **global thread index formula** — how each thread figures out which array element it owns.

> `__global__` 关键字表示它在 GPU 上运行。`int i = blockIdx.x * blockDim.x + threadIdx.x` 这一行是**全局线程索引公式**——每个线程用它算出自己负责哪个数组元素。

Picture the GPU launching thousands of threads organized into blocks:

> 想象 GPU 启动成千上万个线程,按块(block)组织:

```
Block 0          Block 1          Block 2
[t0 t1 t2 t3]   [t4 t5 t6 t7]   [t8 t9 t10 t11]
```

> (图示:线程按块组织,Block 0/1/2 各含 4 个线程)

- `blockDim.x` = threads per block (4 here)
- `blockIdx.x` = which block this thread is in (0, 1, 2)
- `threadIdx.x` = position within the block (0–3)

> - `blockDim.x` = 每块的线程数(此处为 4)
> - `blockIdx.x` = 该线程位于哪个块(0、1、2)
> - `threadIdx.x` = 块内位置(0–3)

So for thread `t7`: `blockIdx.x=1`, `blockDim.x=4`, `threadIdx.x=3` → `1*4 + 3 = 7`. That thread processes `arr[7]`.

> 以线程 `t7` 为例:`blockIdx.x=1`、`blockDim.x=4`、`threadIdx.x=3` → `1*4 + 3 = 7`,该线程处理 `arr[7]`。

**Infrastructure analogy:** It's like running a bash one-liner via `parallel` across 4096 workers simultaneously. Each worker knows its index (like `$PARALLEL_SEQ`) and processes its slice.

> **基础设施类比:**就像用 `parallel` 把一条 bash 命令同时分给 4096 个 worker。每个 worker 知道自己的编号(类似 `$PARALLEL_SEQ`),各自处理自己那一片。

In PyTorch, every operation (add, multiply, matmul, softmax) dispatches one or more CUDA kernels. You don't write them — but they're there.

> 在 PyTorch 里,每个操作(加、乘、matmul、softmax)都会分发一个或多个 CUDA kernel。你不用亲手写——但它们一直在那里。

In [2]:
# Demonstrate kernel dispatch: every PyTorch op is a kernel(演示 kernel 分发:每个 PyTorch 操作都是一个 kernel)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Create a simple tensor on GPU(在 GPU 上创建一个简单张量)
x = torch.ones(1024, 1024, dtype=torch.float32, device=device)

# Each of these dispatches a CUDA kernel(以下每个操作都会分发一个 CUDA kernel)
ops = {
    "Elementwise multiply (x*2)": lambda t: t * 2,
    "ReLU activation":            lambda t: torch.relu(t - 0.5),
    "LayerNorm":                   lambda t: torch.nn.functional.layer_norm(t, t.shape[-1:]),
    "SiLU activation":             lambda t: torch.nn.functional.silu(t),
    "Softmax":                     lambda t: torch.softmax(t, dim=-1),
}

def time_op(fn, tensor, n=100):
    # Warmup(预热)
    for _ in range(5):
        fn(tensor)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n):
        fn(tensor)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    return (time.perf_counter() - start) / n * 1000  # ms(毫秒)

print(f"\nKernel dispatch times (1024x1024 tensor on {device}):")
print(f"{'Operation':<35} {'Latency (ms)':>12}")
print("-" * 50)
for name, fn in ops.items():
    ms = time_op(fn, x)
    print(f"{name:<35} {ms:>10.3f} ms")

Device: cuda

Kernel dispatch times (1024x1024 tensor on cuda):
Operation                           Latency (ms)
--------------------------------------------------
Elementwise multiply (x*2)               0.069 ms
ReLU activation                          0.122 ms
LayerNorm                                0.087 ms
SiLU activation                          0.043 ms
Softmax                                  0.025 ms


## Part 2: The Cost of Separate Kernels (Kernel Launch Overhead + Memory Round-trips)(第二部分:分离 kernel 的代价——启动开销 + 内存往返)

When two kernels run back-to-back on the same data, there are two costs:

> 两个 kernel 在同一份数据上首尾相接运行时,会产生两笔开销:

1. **Kernel launch overhead:** Each kernel launch has a small but non-zero overhead (~10-50 microseconds). If you're running hundreds of operations per token, this adds up.

2. **Memory round-trips:** The naive pipeline writes intermediate results to GPU VRAM and reads them back. From Day 04, we know memory access is often the bottleneck.

> 1. **kernel 启动开销:**每次启动 kernel 都有一笔小而非零的开销(约 10–50 微秒)。若每个词元要跑几百个操作,累积起来相当可观。
>
> 2. **内存往返:**朴素流水线会把中间结果写进显存、再读回来。由 Day 04 可知,内存访问常常就是瓶颈。

**Example from the book:** Two kernels: `multiply_by_2` and `multiply_by_3` running sequentially:
1. Read `[1, 2, 3]` from memory
2. Run `×2` → output `[2, 4, 6]`
3. **Write `[2, 4, 6]` to memory** ← wasted I/O
4. **Read `[2, 4, 6]` from memory** ← wasted I/O
5. Run `×3` → output `[6, 12, 18]`
6. Write `[6, 12, 18]` to memory

> **书中的例子:**两个 kernel——`multiply_by_2` 与 `multiply_by_3` 顺序执行:
> 1. 从内存读 `[1, 2, 3]`
> 2. 执行 `×2` → 输出 `[2, 4, 6]`
> 3. **把 `[2, 4, 6]` 写回内存** ← 浪费的 I/O
> 4. **再从内存读回 `[2, 4, 6]`** ← 浪费的 I/O
> 5. 执行 `×3` → 输出 `[6, 12, 18]`
> 6. 把 `[6, 12, 18]` 写回内存

With a **fused kernel** (`multiply_by_6`):
1. Read `[1, 2, 3]` from memory
2. Run `×6` → output `[6, 12, 18]`
3. Write `[6, 12, 18]` to memory

> 换成**融合 kernel**(`multiply_by_6`)后:
> 1. 从内存读 `[1, 2, 3]`
> 2. 执行 `×6` → 输出 `[6, 12, 18]`
> 3. 把 `[6, 12, 18]` 写回内存

Steps 3 and 4 are eliminated — 2 fewer memory operations.

> 原来的第 3、4 步消失了——少了两趟内存操作。

In [3]:
# Demonstrate memory round-trip cost with a real example(用真实例子演示内存往返的开销)
# We'll simulate a typical LLM FFN activation: linear -> SiLU -> linear(模拟典型 LLM FFN 激活:linear -> SiLU -> linear)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Scaled down from Llama-7B (4096/11008) to fit DGX Spark memory(从 Llama-7B 的 4096/11008 缩小,以适配 DGX Spark 显存)
batch_seq = 16   # batch * seq_len tokens(批大小 × 序列长度的词元数)
hidden = 2048
ffn_dim = 5504

x = torch.randn(batch_seq, hidden, dtype=torch.float16, device=device)
W1 = torch.randn(hidden, ffn_dim, dtype=torch.float16, device=device)
W2 = torch.randn(ffn_dim, hidden, dtype=torch.float16, device=device)
W_gate = torch.randn(hidden, ffn_dim, dtype=torch.float16, device=device)

# Unfused: separate operations, each kernel writes/reads intermediate(未融合:各操作独立,每个 kernel 都要写/读中间结果)
def ffn_unfused(x, W1, W_gate, W2):
    hidden_states = torch.matmul(x, W1)      # kernel 1: matmul(kernel 1:矩阵乘)
    gate = torch.matmul(x, W_gate)           # kernel 2: matmul(kernel 2:矩阵乘)
    gate_activated = torch.nn.functional.silu(gate)  # kernel 3: silu(kernel 3:SiLU 激活)
    hidden_gated = hidden_states * gate_activated     # kernel 4: elementwise mul(kernel 4:逐元素乘)
    out = torch.matmul(hidden_gated, W2)     # kernel 5: matmul(kernel 5:矩阵乘)
    return out

def benchmark(fn, n=50):
    for _ in range(5):
        fn()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n):
        fn()
        if device.type == 'cuda':
            torch.cuda.synchronize()
    return (time.perf_counter() - start) / n * 1000

t_unfused = benchmark(lambda: ffn_unfused(x, W1, W_gate, W2))
print(f"FFN unfused (5 kernel dispatches): {t_unfused:.3f} ms")

# torch.compile can automatically fuse kernels(torch.compile 能自动融合 kernel)
if device.type == 'cuda':
    try:
        ffn_compiled = torch.compile(ffn_unfused, mode='reduce-overhead')
        # Warmup compilation(预热编译)
        _ = ffn_compiled(x, W1, W_gate, W2)
        torch.cuda.synchronize()
        t_compiled = benchmark(lambda: ffn_compiled(x, W1, W_gate, W2))
        print(f"FFN compiled (torch.compile):    {t_compiled:.3f} ms")
        speedup = t_unfused / t_compiled
        print(f"Speedup from compilation:        {speedup:.2f}x")
    except Exception as e:
        print(f"torch.compile not available: {e}")
else:
    print("GPU not available, skipping compile comparison")

# Free memory for subsequent cells(为后续单元释放显存)
del x, W1, W2, W_gate
if device.type == 'cuda':
    torch.cuda.empty_cache()


FFN unfused (5 kernel dispatches): 0.491 ms


W0409 15:06:17.965000 638165 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


FFN compiled (torch.compile):    1.375 ms
Speedup from compilation:        0.36x


## Part 3: Kernel Selection — Choosing the Right Implementation(第三部分:kernel 选择——挑对实现)

**Kernel selection** is choosing the best kernel implementation for a specific operation on specific hardware.

> **kernel 选择(kernel selection)**,就是为特定硬件上的特定操作挑选最好的 kernel 实现。

Why does this matter? Kernels are hardware-specific. A kernel optimized for H100's Hopper architecture uses different Tensor Core layouts than one for A100. A kernel that hard-codes VRAM bandwidth values for one GPU will underperform on another.

> 这为什么重要?kernel 与硬件强相关。为 H100 Hopper 架构优化的 kernel,其 Tensor Core 布局与 A100 版本不同;把某块 GPU 的显存带宽硬编码进去的 kernel,换一块卡就会性能不佳。

**Common specialized kernels:**

> **常见的专用 kernel:**

| Kernel library | What it provides |
|---|---|
| **cuBLAS** | Standard GEMM (matrix multiply) for all NVIDIA GPUs |
| **CUTLASS** | Template library for writing custom GEMM variants |
| **FlashAttention** | Tiled attention that avoids writing S and P matrices to memory |
| **DeepGEMM** | Optimized FP8 GEMM kernels for Hopper GPUs (released by DeepSeek) |
| **FlashInfer** | Optimized attention + sampling kernels for LLM inference |

> (表中:cuBLAS——所有 NVIDIA GPU 通用的标准 GEMM(矩阵乘);CUTLASS——编写自定义 GEMM 变体的模板库;FlashAttention——分块计算注意力,避免把 S、P 矩阵写入显存;DeepGEMM——面向 Hopper GPU 的 FP8 GEMM 优化 kernel(DeepSeek 发布);FlashInfer——面向 LLM 推理的注意力 + 采样优化 kernel)

**Infrastructure analogy:** This is like choosing between `nginx`, `caddy`, and `apache` for your web server. They all serve HTTP, but each has specific performance characteristics for different workloads. You pick based on your use case.

> **基础设施类比:**这就像在 `nginx`、`caddy`、`apache` 之间选 Web 服务器:都能服务 HTTP,但各自的性能特性适配不同负载。按你的使用场景来挑。

In [4]:
# Demonstrate kernel selection: PyTorch automatically selects the best
# attention kernel based on hardware. We can compare standard vs flash attention.(演示 kernel 选择:PyTorch 会根据硬件自动选择最优 attention kernel。我们来对比标准注意力与 FlashAttention。)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def standard_attention(Q, K, V, scale):
    """Standard attention: writes S and P matrices to memory.(标准注意力:会把 S、P 矩阵写入显存。)"""
    S = torch.matmul(Q, K.transpose(-2, -1)) * scale  # NxN intermediate(N×N 中间矩阵)
    P = torch.softmax(S, dim=-1)                        # NxN intermediate(N×N 中间矩阵)
    O = torch.matmul(P, V)
    return O

def flash_attention_pytorch(Q, K, V, scale):
    """Uses PyTorch's SDPA which selects FlashAttention when available.(使用 PyTorch 的 SDPA,可用时自动选用 FlashAttention。)"""
    return torch.nn.functional.scaled_dot_product_attention(Q, K, V, scale=scale)

# Sized to fit DGX Spark memory — scale up on larger GPUs(尺寸按 DGX Spark 显存裁剪——更大的 GPU 可调大)
batch, heads, d = 2, 4, 64
seq_lengths = [128, 256, 512, 1024]

print(f"Attention benchmark (batch={batch}, heads={heads}, d={d})")
print(f"{'Seq len':>8} {'Standard (ms)':>14} {'Flash/SDPA (ms)':>16} {'Speedup':>8} {'Memory saved':>12}")
print("-" * 65)

for seq_len in seq_lengths:
    Q = torch.randn(batch, heads, seq_len, d, dtype=torch.float16, device=device)
    K = torch.randn(batch, heads, seq_len, d, dtype=torch.float16, device=device)
    V = torch.randn(batch, heads, seq_len, d, dtype=torch.float16, device=device)
    scale = d ** -0.5

    def bench(fn, n=20):
        for _ in range(3): fn(Q, K, V, scale)
        if device.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(n):
            fn(Q, K, V, scale)
            if device.type == 'cuda': torch.cuda.synchronize()
        return (time.perf_counter() - t) / n * 1000

    t_std = bench(standard_attention)
    t_flash = bench(flash_attention_pytorch)
    speedup = t_std / t_flash

    # Memory: standard stores NxN matrix (2 of them: S and P)(内存:标准实现要存 N×N 矩阵(S 和 P 各一个))
    n2_mem_mb = 2 * batch * heads * seq_len * seq_len * 2 / 1e6  # FP16(FP16,每值 2 字节)
    print(f"{seq_len:>8} {t_std:>12.3f}ms {t_flash:>14.3f}ms {speedup:>7.2f}x {n2_mem_mb:>10.1f} MB")

    del Q, K, V  # free between iterations(迭代之间释放)

if device.type == 'cuda':
    torch.cuda.empty_cache()


Attention benchmark (batch=2, heads=4, d=64)
 Seq len  Standard (ms)  Flash/SDPA (ms)  Speedup Memory saved
-----------------------------------------------------------------
     128        0.097ms          0.040ms    2.41x        0.5 MB
     256        0.158ms          0.076ms    2.07x        2.1 MB
     512        0.232ms          0.032ms    7.26x        8.4 MB
    1024        0.677ms          0.186ms    3.63x       33.6 MB


## Part 4: Profiling Kernel Dispatch with torch.profiler(第四部分:用 torch.profiler 分析 kernel 分发)

You can't optimize what you can't measure. PyTorch's built-in profiler shows you exactly which kernels are running and how long each takes.

> 测不了,就优化不了。PyTorch 内置的 profiler 能精确告诉你哪些 kernel 在跑、各花多久。

In [5]:
from torch.profiler import profile, record_function, ProfilerActivity

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
activities = [ProfilerActivity.CPU]
if device.type == 'cuda':
    activities.append(ProfilerActivity.CUDA)

batch, seq, hidden = 4, 128, 2048
x = torch.randn(batch, seq, hidden, dtype=torch.float16, device=device)
W = torch.randn(hidden, hidden * 4, dtype=torch.float16, device=device)

with profile(activities=activities, record_shapes=True) as prof:
    with record_function("FFN_forward"):
        # Simulate a small FFN block(模拟一个小型 FFN 块)
        h = torch.matmul(x, W)                              # linear(线性层)
        h = torch.nn.functional.silu(h)                     # activation(激活)
        h = torch.matmul(h, W.t()[:h.shape[-1]])            # project back(投影回去)
        h = torch.nn.functional.layer_norm(h, [h.shape[-1]]) # norm(归一化)

# Print top operations by CUDA time(按 CUDA 耗时打印最耗时的操作)
print("Top operations (sorted by self time):")
print("-" * 80)
table = prof.key_averages().table(
    sort_by="self_cuda_time_total" if device.type == 'cuda' else "self_cpu_time_total",
    row_limit=10
)
print(table)

Top operations (sorted by self time):
--------------------------------------------------------------------------------
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            FFN_forward         0.00%       0.000us         0.00%       0.000us       0.000us      11.561ms      2222.48%      11.561ms      11.561ms             1  
                                  Lazy Function Loading         0.40%      57.984us     

/home/nvidia/src/github.com/elizabetht/100-days-of-inference/.venv/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


## Try These Experiments(动手实验)

1. **Measure kernel launch overhead:** Create a loop that launches 1000 tiny elementwise operations (e.g., `x + 1`) on a small tensor. Compare the total time to doing a single equivalent operation. The difference is accumulated launch overhead.

2. **Flash attention memory savings:** In the `standard_attention` function, print the size of the intermediate `S` matrix at different sequence lengths. Calculate how many GB it would consume for a full 32K context window (32768 tokens) with 32 heads.

3. **torch.compile impact:** Try adding `@torch.compile` to `ffn_unfused` for a larger matrix (hidden=8192, ffn=28672, batch_seq=256). Compare latency before and after compilation. What percentage speedup do you get?

> 1. **测量 kernel 启动开销:**写一个循环,在小张量上启动 1000 次微型逐元素操作(如 `x + 1`),把总耗时与单次等价操作对比,差值就是累积的启动开销。
>
> 2. **FlashAttention 的内存节省:**在 `standard_attention` 函数中,打印不同序列长度下中间矩阵 `S` 的大小。算一算:完整 32K 上下文窗口(32768 个词元)、32 个头时,它要占用多少 GB?
>
> 3. **torch.compile 的影响:**对更大的矩阵(hidden=8192、ffn=28672、batch_seq=256)给 `ffn_unfused` 加上 `@torch.compile`,对比编译前后的延迟。能提速百分之多少?

## Key Takeaways(核心要点)

- A CUDA **kernel** is a parallel function running on thousands of GPU threads simultaneously — every PyTorch operation dispatches one or more kernels.
- **Kernel fusion** eliminates intermediate memory reads/writes between sequential operations — critical for memory-bound workloads like decode.
- **Kernel selection** picks the best implementation for the hardware. FlashAttention, cuBLAS, and DeepGEMM are specialized for different tasks and GPUs.
- `torch.compile` performs automatic kernel fusion for some patterns; complex kernels like FlashAttention require handwritten fused implementations.
- **What's next:** Day 08 — PyTorch, model file formats, ONNX & TensorRT: how models are stored, compiled, and optimized for deployment.

> - CUDA **kernel** 是由数千个 GPU 线程同时并行运行的函数——每个 PyTorch 操作都会分发一个或多个 kernel。
> - **kernel 融合**消除顺序操作之间的中间内存读写——对 decode 这类带宽受限负载至关重要。
> - **kernel 选择**为硬件挑最优实现。FlashAttention、cuBLAS、DeepGEMM 各自面向不同的任务和 GPU。
> - `torch.compile` 能对某些模式自动做 kernel 融合;FlashAttention 这类复杂 kernel 仍需手写融合实现。
> - **下一步:**Day 08——PyTorch、模型文件格式、ONNX 与 TensorRT:模型如何存储、编译并针对部署做优化。

## References(参考资料)
- *Inference Engineering* Ch 4.1 (pp. 96–101) — CUDA kernels, kernel selection, kernel fusion

> - 《Inference Engineering》第 4.1 章(第 96–101 页)——CUDA kernel、kernel 选择、kernel 融合